# Livestock Adaptive Feed Procurement — Next Stage

**Purpose:** inspect the preprocessed files, build a clearly labelled *illustrative* feed-mix linear program, and test inventory resilience under a supply disruption.

## Data and safety notes
- Input files are from `livestock_results.zip` generated in the previous notebook.
- The mandi dataset is a one-day snapshot (18 Oct 2026), not a historical time series.
- The source feed-nutrient library contains records from multiple countries and inconsistent units. This notebook does **not** silently join market prices to feed nutrients.
- The optimizer below uses **synthetic illustrative ingredient values, supplier prices, inventory, and demand**. It demonstrates the method only; it is not a cattle-feeding recommendation. Replace assumptions with locally validated values and expert-reviewed requirements before making real decisions.


In [ ]:
from pathlib import Path
import zipfile, os, shutil
import pandas as pd
import numpy as np

INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/livestock_optimizer")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Locate the uploaded results ZIP. If you've attached the individual CSVs instead,
# the fallback searches for those CSVs directly.
zip_candidates = list(INPUT_DIR.rglob("livestock_results.zip"))
if zip_candidates:
    results_zip = zip_candidates[0]
    print("Using ZIP:", results_zip)
    with zipfile.ZipFile(results_zip) as z:
        z.extractall(WORK_DIR / "results")
    DATA_DIR = WORK_DIR / "results"
else:
    print("No livestock_results.zip found; searching for individual result CSVs.")
    DATA_DIR = INPUT_DIR

def locate_file(filename):
    hits = list(DATA_DIR.rglob(filename))
    if not hits and DATA_DIR != INPUT_DIR:
        hits = list(INPUT_DIR.rglob(filename))
    return hits[0] if hits else None

for name in ["feed_nutrients_clean_long.csv", "feed_nutrients_clean_wide.csv",
             "market_prices_feed_candidates.csv", "ingredient_crosswalk_review_required.csv"]:
    print(name, "=>", locate_file(name))


In [ ]:
LONG_PATH = locate_file("feed_nutrients_clean_long.csv")
WIDE_PATH = locate_file("feed_nutrients_clean_wide.csv")
MARKET_PATH = locate_file("market_prices_feed_candidates.csv")
CROSSWALK_PATH = locate_file("ingredient_crosswalk_review_required.csv")

if LONG_PATH is None or MARKET_PATH is None:
    raise FileNotFoundError(
        "Required cleaned CSVs not found. Attach livestock_results.zip (or the cleaned CSVs) "
        "using Kaggle's Add Input, then rerun this notebook."
    )

nutrients_long = pd.read_csv(LONG_PATH, low_memory=False)
market = pd.read_csv(MARKET_PATH, low_memory=False)
nutrients_wide = pd.read_csv(WIDE_PATH, low_memory=False) if WIDE_PATH else pd.DataFrame()
crosswalk = pd.read_csv(CROSSWALK_PATH) if CROSSWALK_PATH else pd.DataFrame()

print("Nutrient long:", nutrients_long.shape)
print("Nutrient wide:", nutrients_wide.shape)
print("Market candidates:", market.shape)
print("\nMarket dates:")
print(market["arrival_date"].value_counts(dropna=False).head(10))
display(market.head())


In [ ]:
# Inspect the nutrient codes and units before any possible merge.
display(
    nutrients_long[nutrients_long["nutrient_code"].isin(["CP", "ME", "GE", "DM"])]
    .groupby(["nutrient_code", "nutrient_unit"], dropna=False)
    .size().rename("records").reset_index()
    .sort_values(["nutrient_code", "records"], ascending=[True, False])
)

print("Crosswalk requiring manual review:")
display(crosswalk)

print("Market commodity/category counts:")
display(market.groupby(["commodity_original", "candidate_feed_category"]).size()
        .rename("records").reset_index().sort_values("records", ascending=False))


## 1. Illustrative ration optimizer (synthetic values)

This linear program minimizes the cost of a daily feed mix subject to:
- total ration mass,
- minimum crude-protein fraction,
- minimum energy level,
- ingredient availability limits.

The ingredient nutrient values and prices below are **demonstration assumptions**, not extracted/validated facts from the source files. They are intentionally separated from the market snapshot because matching commodity names alone is not enough to establish equivalent feed form, moisture basis, or price units.


In [ ]:
from scipy.optimize import linprog

# Synthetic demonstration profile ONLY. Replace after validating feed composition,
# units, animal requirements, and local supplier quotes.
ingredients = pd.DataFrame([
    {"ingredient": "dry_fodder_demo", "cost_inr_per_kg": 8.0,  "cp_fraction": 0.06, "energy_mj_per_kg": 6.0,  "max_kg_day": 7.0},
    {"ingredient": "green_fodder_demo", "cost_inr_per_kg": 5.0, "cp_fraction": 0.12, "energy_mj_per_kg": 7.0, "max_kg_day": 5.0},
    {"ingredient": "maize_demo", "cost_inr_per_kg": 24.0, "cp_fraction": 0.09, "energy_mj_per_kg": 13.0, "max_kg_day": 4.0},
    {"ingredient": "protein_supplement_demo", "cost_inr_per_kg": 38.0, "cp_fraction": 0.40, "energy_mj_per_kg": 10.0, "max_kg_day": 2.0},
])

TOTAL_RATION_KG = 10.0
MIN_CP_FRACTION = 0.12
MIN_ENERGY_MJ_PER_KG = 8.0

# linprog uses A_ub @ x <= b_ub. Reverse the minimum constraints.
c = ingredients["cost_inr_per_kg"].to_numpy()
A_ub = np.vstack([
    -ingredients["cp_fraction"].to_numpy(),
    -ingredients["energy_mj_per_kg"].to_numpy()
])
b_ub = np.array([
    -MIN_CP_FRACTION * TOTAL_RATION_KG,
    -MIN_ENERGY_MJ_PER_KG * TOTAL_RATION_KG
])
A_eq = np.ones((1, len(ingredients)))
b_eq = np.array([TOTAL_RATION_KG])
bounds = [(0, float(v)) for v in ingredients["max_kg_day"]]

result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                 bounds=bounds, method="highs")

if not result.success:
    print("Optimizer did not find a feasible ration:", result.message)
else:
    ingredients["optimized_kg_day"] = result.x
    ingredients["daily_cost_inr_demo"] = ingredients["optimized_kg_day"] * ingredients["cost_inr_per_kg"]
    total_cp_kg = float((ingredients["optimized_kg_day"] * ingredients["cp_fraction"]).sum())
    total_energy_mj = float((ingredients["optimized_kg_day"] * ingredients["energy_mj_per_kg"]).sum())
    print("Status:", result.message)
    print(f"Illustrative daily cost: ₹{result.fun:,.2f} (synthetic prices)")
    print(f"Ration mass: {ingredients['optimized_kg_day'].sum():.2f} kg/day")
    print(f"Crude protein: {100*total_cp_kg/TOTAL_RATION_KG:.2f}% (illustrative)")
    print(f"Energy: {total_energy_mj/TOTAL_RATION_KG:.2f} MJ/kg (illustrative)")
    display(ingredients[["ingredient", "optimized_kg_day", "cost_inr_per_kg", "daily_cost_inr_demo"]])


## 2. Inventory projection and reorder-point demonstration

This section uses synthetic starting stock, daily consumption, supplier lead time, and safety stock to illustrate how the system can detect a likely stockout and recommend a reorder trigger. These values are not actual farm records.


In [ ]:
# Synthetic inventory scenario; change these assumptions for experiments.
inventory = pd.DataFrame([
    {"ingredient": "dry_fodder_demo", "stock_kg": 120.0, "daily_use_kg": 4.0, "lead_time_days": 3, "safety_stock_days": 2, "order_qty_kg": 60.0},
    {"ingredient": "green_fodder_demo", "stock_kg": 45.0, "daily_use_kg": 3.0, "lead_time_days": 1, "safety_stock_days": 1, "order_qty_kg": 30.0},
    {"ingredient": "maize_demo", "stock_kg": 35.0, "daily_use_kg": 2.0, "lead_time_days": 5, "safety_stock_days": 3, "order_qty_kg": 40.0},
    {"ingredient": "protein_supplement_demo", "stock_kg": 12.0, "daily_use_kg": 1.0, "lead_time_days": 4, "safety_stock_days": 2, "order_qty_kg": 15.0},
])
inventory["reorder_point_kg"] = inventory["daily_use_kg"] * (
    inventory["lead_time_days"] + inventory["safety_stock_days"]
)
inventory["days_of_stock"] = inventory["stock_kg"] / inventory["daily_use_kg"]
inventory["reorder_now"] = inventory["stock_kg"] <= inventory["reorder_point_kg"]
display(inventory)


In [ ]:
# Simulate 30 days with a simple reorder policy. Synthetic delivery quantities arrive
# after the supplier lead time. This is a baseline, not the full multi-period MILP.
HORIZON_DAYS = 30
simulation_rows = []

for _, item in inventory.iterrows():
    stock = float(item["stock_kg"])
    pending = []  # list of (arrival_day, quantity)
    reorder_point = float(item["reorder_point_kg"])
    for day in range(1, HORIZON_DAYS + 1):
        arrivals = sum(q for arrival_day, q in pending if arrival_day == day)
        pending = [(arrival_day, q) for arrival_day, q in pending if arrival_day != day]
        stock += arrivals
        stock -= float(item["daily_use_kg"])
        stockout = stock < -1e-9
        if stock < 0:
            stock = 0.0
        order_placed = False
        if stock <= reorder_point and not any(arrival_day > day for arrival_day, _ in pending):
            pending.append((day + int(item["lead_time_days"]), float(item["order_qty_kg"])))
            order_placed = True
        simulation_rows.append({
            "day": day, "ingredient": item["ingredient"], "ending_stock_kg": stock,
            "stockout_event": stockout, "order_placed": order_placed,
            "scheduled_arrivals_kg": arrivals
        })

simulation = pd.DataFrame(simulation_rows)
display(simulation.head(15))
summary = simulation.groupby("ingredient").agg(
    minimum_stock_kg=("ending_stock_kg", "min"),
    stockout_days=("stockout_event", "sum"),
    orders_placed=("order_placed", "sum")
).reset_index()
display(summary)


## 3. Export results

Save the illustrative ration solution (if feasible), inventory triggers, daily simulation, and the input market snapshot to one ZIP for your project folder.


In [ ]:
EXPORT_DIR = Path("/kaggle/working/livestock_optimizer/results")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

market.to_csv(EXPORT_DIR / "market_price_candidates_snapshot.csv", index=False)
nutrients_long.head(0).to_csv(EXPORT_DIR / "nutrient_schema.csv", index=False)
inventory.to_csv(EXPORT_DIR / "inventory_reorder_points_synthetic.csv", index=False)
simulation.to_csv(EXPORT_DIR / "inventory_simulation_synthetic.csv", index=False)
summary.to_csv(EXPORT_DIR / "inventory_simulation_summary_synthetic.csv", index=False)

if result.success:
    ingredients.to_csv(EXPORT_DIR / "illustrative_ration_solution_synthetic.csv", index=False)

with open(EXPORT_DIR / "README.txt", "w", encoding="utf-8") as f:
    f.write(
        "Livestock optimizer stage outputs.\n"
        "IMPORTANT: ration ingredient values, prices, inventory, demand, lead times, and order sizes "
        "are synthetic illustrative assumptions. These outputs are not real cattle-feeding advice.\n"
        "The market CSV is a one-day snapshot and has not been automatically merged to nutrient data.\n"
        "Validate feed identities, nutrient units, price units, Indian cattle requirements, and supplier data "
        "before operational use.\n"
    )

ZIP_OUT = "/kaggle/working/livestock_optimizer_results.zip"
shutil.make_archive(ZIP_OUT.removesuffix(".zip"), "zip", root_dir=EXPORT_DIR)
print("Created:", ZIP_OUT, "| size MB:", round(os.path.getsize(ZIP_OUT)/(1024*1024), 2))
from IPython.display import FileLink, display
display(FileLink(ZIP_OUT))
